In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Direct path to your dataset file
file_path = r"C:\Users\AI_LAB\Downloads\archive\senate_stock_discosures.csv"

# Load dataset
df = pd.read_csv(file_path)

# ======================================================
# PART 1: Initial Data Cleaning & Basic Exploration
# ======================================================

# Remove duplicate rows[cite: 1]
df = df.drop_duplicates()

# Remove leading/trailing whitespaces from text columns[cite: 1]
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip()

# Fill missing values (Mean for numerical, Mode for categorical)[cite: 1]
for col in df.select_dtypes(include='number').columns:
    df[col] = df[col].fillna(df[col].mean())

for col in df.select_dtypes(include='object').columns:
    if not df[col].mode().empty:
        df[col] = df[col].fillna(df[col].mode()[0])

print("\n===== Cleaned (Crisp) Dataset =====")
print(df)

print("\n===== First 10 Rows =====")
print(df.head(10))

print("\n===== Dataset Info =====")
print(df.info())

print("\n===== Dataset Summary Statistics =====")
print(df.describe(include='all'))


# ======================================================
# PART 2: KDD / SEMMA Pipeline Execution
# ======================================================

# 1. Sample (Shuffle dataset)[cite: 1]
df_sample = df.sample(frac=1, random_state=42).copy()

# 2. Explore[cite: 1]
print("\n===== Sampled Data Info =====")
print(df_sample.info())
print(df_sample.describe(include='all'))

# 3. Modify (Clean & Preprocess)[cite: 1]
for col in df_sample.select_dtypes(include='number').columns:
    df_sample[col] = SimpleImputer(strategy='mean').fit_transform(df_sample[[col]])

for col in df_sample.select_dtypes(include='object').columns:
    df_sample[col] = SimpleImputer(strategy='most_frequent').fit_transform(df_sample[[col]]).ravel()

# Encode categorical variables[cite: 1]
for col in df_sample.select_dtypes(include='object').columns:
    df_sample[col] = LabelEncoder().fit_transform(df_sample[col])

# Feature / Target Separation
# Targets the transaction type column if present, otherwise defaults to the last column[cite: 1]
if 'type' in df_sample.columns:
    X = df_sample.drop(columns=['type'])
    y = df_sample['type']
elif 'transaction' in df_sample.columns:
    X = df_sample.drop(columns=['transaction'])
    y = df_sample['transaction']
else:
    X = df_sample.iloc[:, :-1]
    y = df_sample.iloc[:, -1]

# Feature Scaling[cite: 1]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/Test Split[cite: 1]
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# 4. Model (Decision Tree Classifier)[cite: 1]
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

# 5. Assess[cite: 1]
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"\nModel Accuracy: {accuracy:.2%}")


===== Cleaned (Crisp) Dataset =====
     ticker                                asset_name asset_type  stock_price  \
0       RTX              RTX Corporation Common Stock      Stock       100.99   
1       DHR                              Danaher Corp      Stock       249.80   
2        UL                          Unilever Plc ADR      Stock        47.99   
3        IR                        Ingersoll Rand Inc      Stock        93.48   
4       NVS                           Novartis Ag ADR      Stock        95.81   
...     ...                                       ...        ...          ...   
4689   AAPL                       Apple Inc. (NASDAQ)      Stock        25.02   
4690    AXP           American Express Company (NYSE)      Stock        80.19   
4691    NOG         Northern Oil and Gas, Inc. (AMEX)      Stock        63.13   
4693    FCX              Freeport-McMoRan Inc. (NYSE)      Stock        21.02   
4697  MDDVX  BlackRock Equity Dividend Inv A (NASDAQ)      Stock        